In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

In [19]:
df_types = pd.read_csv("transaction_types.csv")
df_txn = pd.read_csv("transactions_data_training.csv")
df_profile = pd.read_csv("user_profile.csv")
df_mcc = pd.read_csv("mcc_group_definition.csv")

In [21]:
df_txn.head()

,user_id,transaction_date,transaction_type,mcc_group,amount_n26_currency,dataset_transaction,dataset_user
0,a78884f5e76951188c1e719d4956773a,2016-02-01,DT,NaN,350,training,training
1,b0333294fef6ff1299102a70ad46b126,2016-02-01,DT,NaN,202,training,training
2,7b8d2a2780adae0cd0c248e92c1b28dc,2016-02-01,DT,NaN,291,training,training
3,cc4abaa500f7db4390ae3f02bd36d805,2016-02-01,DT,NaN,214,training,training
4,49b99d1d5ba028566639e8b3eb7c055b,2016-02-01,DT,NaN,272,training,training


In [4]:
df_mcc.head()

,mcc_group,explanation
0,1,ATM
1,2,Groceries
2,3,Gastronomie
3,4,Shopping/(Fashion)
4,5,"Home and DIY, House Energy"


In [22]:
df_profile.head()

,user_id,age,age_band,country,city_tier,employment_status,industry,income_band,account_tenure_days
0,000295594379774ab9ac2c78c946d615,47,45-54,DE,2,self_employed,healthcare,50-75k,119
1,000aa792d73dd82b16a29692772d395a,61,55+,AT,1,employed_ft,finance,50-75k,134
2,0011103fe4ba1264342882b7ab98c641,58,55+,FR,1,retired,other,50-75k,109
3,001679c77c33d7efabf800596fb2a978,53,45-54,DE,2,self_employed,education,50-75k,127
4,001e72a2f4c4376f10011735dde39cd5,28,25-34,IE,2,employed_ft,healthcare,30-50k,94


In [23]:
df_types.head()

,type,explanation,direction,agent
0,AR,Authorization reject,-,Card
1,AE,Authorization expired,In,Card
2,AV,Authorization Reversal,In,Card
3,BBU,Cash26 deposit,In,Partner
4,CT,Credit Transfer,In,Bank Account


In [24]:
df_txn.dtypes

user_id                    str
transaction_date           str
transaction_type           str
mcc_group              float64
amount_n26_currency      int64
dataset_transaction        str
dataset_user               str
dtype: object

In [30]:
df_txn["transaction_date"] = pd.to_datetime(df_txn["transaction_date"])

In [31]:
df_txn.dtypes

user_id                           str
transaction_date       datetime64[us]
transaction_type                  str
mcc_group                     float64
amount_n26_currency             int64
dataset_transaction               str
dataset_user                      str
dtype: object

In [32]:
print(f"Total transaction :" ,df_txn.shape[0], 
      f", User :", df_txn["user_id"].nunique())

Total transaction : 408546 , User : 10000


In [33]:
df_profile.shape

(9470, 9)

In [37]:
direction_map = (
    df_types.set_index("type")["direction"].to_dict()
)

In [38]:
direction_map

{'AR': '-',
 'AE': 'In',
 'AV': 'In',
 'BBU': 'In',
 'CT': 'In',
 'DI': 'In',
 'DR': 'In',
 'PF': 'In',
 'PR': 'In',
 'WU': 'In',
 'AA': 'Out',
 'BUB': 'Out',
 'DD': 'Out',
 'DT': 'Out',
 'FT': 'Out',
 'PT': 'Out',
 'TUB': 'Out',
 'TBU': 'In',
 'WEE': 'Out',
 'AL': 'In',
 'AU': 'Out',
 'BUS': 'Out',
 'BRU': 'Out'}

In [39]:
df_txn["direction"] = df_txn["transaction_type"].map(direction_map)

In [41]:
df_txn.head()

,user_id,transaction_date,transaction_type,mcc_group,amount_n26_currency,dataset_transaction,dataset_user,direction
0,a78884f5e76951188c1e719d4956773a,2016-02-01,DT,NaN,350,training,training,Out
1,b0333294fef6ff1299102a70ad46b126,2016-02-01,DT,NaN,202,training,training,Out
2,7b8d2a2780adae0cd0c248e92c1b28dc,2016-02-01,DT,NaN,291,training,training,Out
3,cc4abaa500f7db4390ae3f02bd36d805,2016-02-01,DT,NaN,214,training,training,Out
4,49b99d1d5ba028566639e8b3eb7c055b,2016-02-01,DT,NaN,272,training,training,Out


In [46]:
df_txn["direction"].isna().sum()

np.int64(0)

In [47]:
df_txn['direction'].value_counts()

direction
Out    336421
In      72125
Name: count, dtype: int64

In [48]:
assert (df_txn['amount_n26_currency'] > 0).all()

In [49]:
print(f"Valid transactions: {len(df_txn):,} (dropped {len(df_txn) - len(df_txn):,} unmapped)")


Valid transactions: 408,546 (dropped 0 unmapped)


In [51]:
df_txn["mcc_group"].isna().sum()

np.int64(150222)

In [52]:
df_profile.columns

Index(['user_id', 'age', 'age_band', 'country', 'city_tier',
       'employment_status', 'industry', 'income_band', 'account_tenure_days'],
      dtype='str')

In [53]:
df_profile_clean = df_profile.drop(
    columns = ["age", "account_tenure_days"]
)

In [54]:
df_profile_clean.head()

,user_id,age_band,country,city_tier,employment_status,industry,income_band
0,000295594379774ab9ac2c78c946d615,45-54,DE,2,self_employed,healthcare,50-75k
1,000aa792d73dd82b16a29692772d395a,55+,AT,1,employed_ft,finance,50-75k
2,0011103fe4ba1264342882b7ab98c641,55+,FR,1,retired,other,50-75k
3,001679c77c33d7efabf800596fb2a978,45-54,DE,2,self_employed,education,50-75k
4,001e72a2f4c4376f10011735dde39cd5,25-34,IE,2,employed_ft,healthcare,30-50k


In [55]:
df_txn['transaction_date'].min(), df_txn['transaction_date'].max()


(Timestamp('2016-02-01 00:00:00'), Timestamp('2016-07-31 00:00:00'))

In [56]:
train_start = "2016-02-01"
train_end ="2016-05-31"
target_start = "2016-06-01"
target_end = "2016-07-31"

In [63]:
df_train_txn = df_txn[(df_txn["transaction_date"] >= train_start) & (df_txn["transaction_date"] <= train_end)]
df_target_txn = df_txn[(df_txn["transaction_date"] >= target_start) & (df_txn["transaction_date"] <= target_end)]


In [64]:
print(
    f"Feature window: {len(df_train_txn):,} txns "
    f"({train_start} to {train_end})"
)
print(
    f"Target window:  {len(df_target_txn):,} txns "
    f"({target_start} to {target_end})"
)

Feature window: 267,593 txns (2016-02-01 to 2016-05-31)
Target window:  140,953 txns (2016-06-01 to 2016-07-31)
